## How to do background Subtraction
- Subtracting two images from each other
- K Nearest Neighbour
- Mixture of Gaussian
- Neural Networks - U Net

### 1. Import Statements

In [3]:
import cv2
import numpy as np

In [17]:
backSub = cv2.createBackgroundSubtractorKNN()
# backSub = cv2.createBackgroundSubtractorMOG2()

### 2. Applying KNN & Gaussian Method

In [22]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print('Unable to Open')
    exit(0)
    
while True:
    ret, frame = cap.read()
    
    kernel = np.ones((3,3), np.uint8)
    
    fgMask = backSub.apply(frame)
    fgMask = cv2.erode(fgMask, kernel, iterations=2)
    fgMask = cv2.dilate(fgMask, kernel, iterations=2)
    
    cv2.rectangle(frame, (10,2), (100,100), (255,255,255), -1)
    cv2.putText(frame, str(cap.get(cv2.CAP_PROP_POS_FRAMES)), (15,15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0))

    # Noise Reduction
    fgMask[np.abs(fgMask) < 250] = 0
    
    cv2.imshow('Frame', frame)
    cv2.imshow('fgMask', fgMask)
    
    if cv2.waitKey(5) & 0xFF == 27:
        break

cap.release() 
cv2.destroyAllWindows()

In [9]:
cap.release() 
cv2.destroyAllWindows()

### 3. Background Subtraction

In [28]:
def resize(dst, img):
    width = img.shape[1]
    height = img.shape[0]
    dim = (width,height)
    resized = cv2.resize(dst, dim, interpolation = cv2.INTER_AREA)
    return resized

In [73]:
cap = cv2.VideoCapture(0)
flower = cv2.VideoCapture('Videos/Video3.mp4')

ret, bgReference = cap.read()

takeBgImage = 0

while True:
    ret1, img = cap.read()
    ret2, bg = flower.read()
    
    if bg is not None:
        bg = resize(bg, bgReference)
        
    if takeBgImage == 0:
        bgReference = img
    
    # Create a Mask
    diff1 = cv2.subtract(img, bgReference)
    diff2 = cv2.subtract(bgReference, img)
    
    diff = diff1 + diff2
    # To Reduce Noise
    diff[abs(diff) < 25.0] = 0
    
    cv2.imshow('diff1', diff)
    
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    # To Reduce Noise
    gray[np.abs(gray) < 10] = 0
    
    fgMask = gray
    
    # Opening 
    kernel = np.ones((3,3), np.uint8)
    
    fgMask = cv2.erode(fgMask, kernel, iterations=2)
    fgMask = cv2.dilate(fgMask, kernel, iterations=2)
    
    fgMask[fgMask>5] = 255
    
    cv2.imshow('Foreground Mask',fgMask)
    
    # Inverting Mask
    fgMask_inv = cv2.bitwise_not(fgMask)
    
    fgImage = cv2.bitwise_and(img, img, mask=fgMask)
    bgImage = cv2.bitwise_and(bg, bg, mask=fgMask_inv)
    
    # Combine the fore- and background images
    bgSub = cv2.add(bgImage, fgImage)
    
    cv2.imshow('Background Removed', bgSub)
    cv2.imshow('Original', img)
    
    key = cv2.waitKey(5) & 0xFF
    if ord('q') == key:
        break
    elif ord('e') == key:
        takeBgImage = 1
        print('Background Captured')
    elif ord('r') == key:
        takeBgImage = 0
        print('Ready to Capture new Background')
        
cap.release()
cv2.destroyAllWindows()

Background Captured


In [70]:
cap.release()
cv2.destroyAllWindows()